# Science Paper Analyzer - туториал


Этот файл объясняет каждый файл проекта простыми словами.


---


## 1. science_paper_analyzer.ipynb


Три шага. Нажать Run три раза.


### Шаг 1: Скачать код


In [ ]:
import subprocess, sys, os, importlib
REPO_URL = "https://github.com/DmitPerson42/science-paper-analyzer.git"
REPO_DIR = "science-paper-analyzer"
if os.path.exists(REPO_DIR):
    import shutil
    shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True, capture_output=True, text=True)
os.chdir(REPO_DIR)
required = ["requests", "pandas", "openpyxl", "lxml", "beautifulsoup4"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True, capture_output=True, text=True)
print("Шаг 1 готов!")


Скачивает код с GitHub. Удаляет старый. Ставит библиотеки.


### Шаг 2: Анализ


In [ ]:
from analyzer import PaperAnalyzer
QUERY = "filtering generative text data language model collapse"
MAX_PER_SOURCE = 5
analyzer = PaperAnalyzer(verbose=True)
papers = analyzer.collect_papers(QUERY, max_per_source=MAX_PER_SOURCE)
if not papers:
    print("Статьи не найдены.")
else:
    df = analyzer.analyze_papers(papers)
    display(df[["title","authors","year","source","overall_score","verdict"]])


collect_papers() ищет по 10 сайтам. analyze_papers() проверяет.


### Шаг 3: Скачать


In [ ]:
if "df" in dir() and len(df) > 0:
    analyzer.export_csv(df, "results.csv")
    from google.colab import files
    files.download("results.csv")
else:
    print("Сначала выполните Шаг 2.")


---


## 2. analyzer.py - мозг программы


Соединяет парсеры, анализаторы, экспортёры.


Streamlit-заглушка: если Streamlit не установлен — создаётся класс-пустышка. Иначе в Colab была бы ошибка.


collect_papers(): проходит по 10 парсерам, пауза 0.3с, удаление дублей.


analyze_papers(): 3 проверки, средний балл. >=0.7 = Real, 0.4-0.7 = Suspicious, <0.4 = Fake.


---


## 3. parsers/


arxiv_parser.py: API arXiv, XML, повторные попытки, extract_year().


semantic_scholar.py: единственный бесплатный источник citationCount.


openreview_parser.py: БЫЛ БАГ! OpenReview возвращает дату в миллисекундах. Старый код давал 1652 вместо 2022. Исправили на extract_year().


aclanthology.py: API + HTML fallback.


jmlr_parser.py: нет API, номер тома = год.


crossref_fallback.py: для ResearchGate, eLibrary, Dissercat, FIPS.


cyberleninka.py: российская библиотека, нет API, нужен User-Agent.


extract_year(): понимает любой формат. Числа, строки, timestamps (в секундах и мс).


---


## 4. analyzers/


citation_analyzer.py: старая статья, 0 цит = 0.2. Новая, 0 цит = 0.5. 1-5 = 0.7. 5-15 = 0.85. >15 = 0.95.


text_analyzer.py: 5 эвристик без нейросетей. AI-маркеры, TTR, повторения, длина предложений, стоп-слова.


journal_analyzer.py: Beall’s List. Доверенные = 0.95, хищнические = 0.15, остальные = 0.6.


---


## 5. exporters/


CSV: utf-8-sig для русских букв в Excel.


Excel: цветная разметка и пояснения.


---


## Как всё работает


Тема -> collect_papers() -> 10 парсеров -> статьи -> analyze_papers() -> 3 проверки -> таблица -> CSV/Excel


Colab: https://colab.research.google.com/github/DmitPerson42/science-paper-analyzer/blob/main/science_paper_analyzer.ipynb


GitHub: https://github.com/DmitPerson42/science-paper-analyzer
